In [31]:
import os
import openai
from dotenv import load_dotenv
import pandas as pd
import numpy as np
load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

In [18]:
from openai import OpenAI
def get_embedding(text: str, api_key: str = OPENROUTER_API_KEY, model: str = "google/gemini-embedding-001") -> list[float]:
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key,
    )   

    embedding = client.embeddings.create(
        model=model,
        input=text,
        encoding_format="float"
    )
    return embedding.data[0].embedding

get_embedding("Hello, world!")

[-0.020920176059007645,
 0.009755219332873821,
 0.004780902992933989,
 -0.05942125245928764,
 0.005082839634269476,
 0.0008645110065117478,
 -0.004584700334817171,
 0.005077085457742214,
 0.01929636485874653,
 0.01731793023645878,
 -0.008134705945849419,
 -0.009040719829499722,
 -0.004410189110785723,
 0.030996358022093773,
 0.10537973791360855,
 0.010174652561545372,
 0.023028461262583733,
 -0.02839803323149681,
 0.0023683798499405384,
 -0.006660488899797201,
 0.0028257493395358324,
 -0.001545771025121212,
 0.03322514519095421,
 -0.008438869379460812,
 0.0032804706133902073,
 0.007675922010093927,
 0.03673441335558891,
 -0.016130663454532623,
 0.02533765323460102,
 0.021699391305446625,
 -0.0008153883391059935,
 0.015893541276454926,
 -0.0450279638171196,
 0.0075905160047113895,
 9.405229502590373e-05,
 0.03321804851293564,
 -0.0026741567999124527,
 -0.014441574923694134,
 -0.004091567825525999,
 0.00832407083362341,
 -0.012123116292059422,
 -0.0029273992404341698,
 0.0070877987891435

In [19]:
df = pd.read_csv("data.csv")
df

,headline,clickbait
0,Should I Get Bings,1
1,Which TV Female Friend Group Do You Belong In,1
2,"The New ""Star Wars: The Force Awakens"" Trailer...",1
3,"This Vine Of New York On ""Celebrity Big Brothe...",1
4,A Couple Did A Stunning Photo Shoot With Their...,1
...,...,...
31995,"To Make Female Hearts Flutter in Iraq, Throw a...",0
31996,"British Liberal Democrat Patsy Calton, 56, die...",0
31997,Drone smartphone app to help heart attack vict...,0
31998,"Netanyahu Urges Pope Benedict, in Israel, to D...",0


In [20]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def embed_column(df: pd.DataFrame, column: str, api_key: str = OPENROUTER_API_KEY, model: str = "google/gemini-embedding-001", max_workers: int = 10) -> pd.DataFrame:
    df_result = df.copy()
    texts = df[column].astype(str).tolist()
    def embed_single(text):
        try:
            return get_embedding(text, api_key=api_key, model=model)
        except Exception as e:
            print(f"Exception on: {text[:50]} -- {str(e)}")
            return None
    
    embeddings = [None] * len(texts)
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_idx = {executor.submit(embed_single, text): idx for idx, text in enumerate(texts)}
        for future in tqdm(as_completed(future_to_idx), total=len(texts), desc=f"Embedding {column}"):
            idx = future_to_idx[future]
            embedding = future.result()
            embeddings[idx] = embedding
    
    embedding_dim = None
    for emb in embeddings:
        if emb is not None:
            embedding_dim = len(emb)
            break
    
    if embedding_dim is None:
        raise ValueError("No successful embeddings found")
    for dim in range(embedding_dim):
        col_name = f"{column}_embedding_{dim}"
        df_result[col_name] = [emb[dim] if emb is not None else np.nan for emb in embeddings]
    
    return df_result

In [21]:
df_emebed = embed_column(df, "headline")

Embedding headline:   3%|▎         | 902/32000 [00:36<17:29, 29.63it/s]

Exception on: How Do You Feel About Period Sex -- 'NoneType' object is not subscriptable


Embedding headline:   9%|▊         | 2741/32000 [01:47<36:49, 13.24it/s]

Exception on: 23 Celebrity Couples We Loved And Lost In 2015 -- Connection error.


Embedding headline:   9%|▊         | 2744/32000 [01:47<43:01, 11.33it/s]

...

Embedding headline:  81%|████████▏ | 26057/32000 [16:49<11:11,  8.85it/s]

Exception on: Honors for a President, but Not Without Debate -- Connection error.
Exception on: Weir Surrenders Early Lead as Glover Charges -- Connection error.


Embedding headline:  81%|████████▏ | 26066/32000 [16:50<07:53, 12.53it/s]

Exception on: Obama to Tell Congress How Agenda Will Aid Economy -- Connection error.
Exception on: Romanian PM and Siemens discuss Electroputere priv -- Connection error.
Exception on: British actress Emily Perry dies at 100 -- Connection error.
Exception on: W.B.C. Turns Teammates Into Rivals, and Vice Versa -- Connection error.


Embedding headline: 100%|██████████| 32000/32000 [20:30<00:00, 26.01it/s]
/var/folders/l0/j_85yd4s3hb97_5f31qtg08r0000gn/T/ipykernel_29239/3442030007.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_result[col_name] = [emb[dim] if emb is not None else np.nan for emb in embeddings]
/var/folders/l0/j_85yd4s3hb97_5f31qtg08r0000gn/T/ipykernel_29239/3442030007.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_result[col_name] = [emb[dim] if emb is not None else np.nan for emb in embeddings]
/var/folders/l0/j_85yd4s3hb97_

In [29]:
df_emebed

,headline,clickbait,headline_embedding_0,headline_embedding_1,headline_embedding_2,headline_embedding_3,headline_embedding_4,headline_embedding_5,headline_embedding_6,headline_embedding_7,...,headline_embedding_3062,headline_embedding_3063,headline_embedding_3064,headline_embedding_3065,headline_embedding_3066,headline_embedding_3067,headline_embedding_3068,headline_embedding_3069,headline_embedding_3070,headline_embedding_3071
0,Should I Get Bings,1,-0.009599,0.004471,-0.006941,-0.061308,0.019271,0.034077,-0.007551,0.005868,...,0.006732,-0.025098,0.006274,0.006979,0.008099,0.003336,-0.026671,0.020475,-0.011650,-0.004918
1,Which TV Female Friend Group Do You Belong In,1,-0.008873,-0.021537,0.020832,-0.046818,-0.033787,0.012369,-0.007303,0.001456,...,0.012202,-0.013303,-0.006872,0.000557,0.010401,-0.001922,-0.012336,-0.022325,-0.028884,0.015051
2,"The New ""Star Wars: The Force Awakens"" Trailer...",1,-0.017801,-0.001513,-0.002053,-0.078880,-0.014149,0.011586,0.001119,-0.007870,...,0.018830,0.006684,0.002229,0.001500,-0.005928,-0.027611,0.009698,0.009091,0.011077,-0.010607
3,"This Vine Of New York On ""Celebrity Big Brothe...",1,-0.020031,-0.036369,0.006938,-0.078386,-0.002978,0.024738,0.005402,-0.001022,...,-0.007846,0.000539,-0.000483,-0.008381,0.008650,-0.002006,0.000524,0.002409,-0.005666,0.002340
4,A Couple Did A Stunning Photo Shoot With Their...,1,-0.036272,0.023537,-0.011256,-0.081985,-0.009544,-0.003775,0.001143,0.028139,...,-0.002253,-0.024292,0.008958,0.015702,0.009919,-0.029299,0.007233,-0.015560,0.004217,0.012071
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31995,"To Make Female Hearts Flutter in Iraq, Throw a...",0,0.008735,-0.011000,0.004503,-0.053650,-0.010739,0.001221,-0.025091,0.033369,...,0.008998,-0.005574,-0.007322,-0.000166,-0.012882,-0.020894,0.008502,0.003487,0.014795,-0.024045
31996,"British Liberal Democrat Patsy Calton, 56, die...",0,-0.001373,0.012539,-0.007904,-0.067004,-0.017387,-0.009129,-0.035517,0.015166,...,0.006833,-0.013549,0.001626,-0.011188,0.012360,-0.004826,-0.012781,0.017501,0.013449,0.002115
31997,Drone smartphone app to help heart attack vict...,0,-0.013575,-0.014688,0.014522,-0.070090,-0.001993,0.008505,0.007275,0.006825,...,0.005997,-0.023342,0.005430,-0.018020,-0.003871,-0.001430,0.010825,-0.007273,0.001120,-0.025126
31998,"Netanyahu Urges Pope Benedict, in Israel, to D...",0,-0.017146,0.035188,0.001730,-0.062668,-0.000364,0.008434,-0.039525,0.013149,...,-0.006111,-0.016955,-0.017481,0.002237,-0.002656,-0.013003,0.023925,-0.007435,0.027725,-0.015822


In [32]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report



In [33]:
# Get all embedding columns
embedding_cols = [col for col in df_emebed.columns if col.startswith('headline_embedding_')]
df_clean = df_emebed.dropna(subset=embedding_cols)
print(f"Rows after removing NaN: {len(df_clean)}/{len(df_emebed)}")

X = df_clean[embedding_cols].values
y = df_clean['clickbait'].values

print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"Clickbait distribution: {pd.Series(y).value_counts().to_dict()}")

Rows after removing NaN: 31947/32000
X shape: (31947, 3072), y shape: (31947,)
Clickbait distribution: {0: 15975, 1: 15972}


In [34]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1
)

model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Evaluate model
accuracy = accuracy_score(y_test, y_pred)
print(f"\nAccuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Clickbait', 'Clickbait']))

Train set: 25557 samples
Test set: 6390 samples

Accuracy: 0.9903

Classification Report:
               precision    recall  f1-score   support

Not Clickbait       0.99      0.99      0.99      3195
    Clickbait       0.99      0.99      0.99      3195

     accuracy                           0.99      6390
    macro avg       0.99      0.99      0.99      6390
 weighted avg       0.99      0.99      0.99      6390



In [38]:
model.save_model('clickbait_model.json')

In [35]:
# Test the model on a new example
test_headline = "You Won't Believe What Happens Next!"
test_embedding = get_embedding(test_headline)

# Convert embedding to the format expected by the model (array of values)
test_embedding_array = np.array([test_embedding])

# Make prediction
prediction = model.predict(test_embedding_array)[0]
prediction_proba = model.predict_proba(test_embedding_array)[0]

print(f"Headline: '{test_headline}'")
print(f"Prediction: {'Clickbait' if prediction == 1 else 'Not Clickbait'}")
print(f"Probability (Not Clickbait): {prediction_proba[0]:.4f}")
print(f"Probability (Clickbait): {prediction_proba[1]:.4f}")


Headline: 'You Won't Believe What Happens Next!'
Prediction: Clickbait
Probability (Not Clickbait): 0.0817
Probability (Clickbait): 0.9183
